# D1: flattened batching and attention metadata

**Goal:** one forward for several independent variable-length prompts, without padding or a model rewrite. Pack IDs, positions and request IDs, then route logits back to each prompt.

**Spec:** correct block-diagonal causal attention; packed logits match separate ours/HF; every request gets its own next token; changing one request cannot influence another.

This is **no-cache packed prefill**. It is not continuous batching or batched cached generation. D2 will introduce paged storage. Runtime validation is pending your run; no speedup is claimed.

Use a **fresh Colab T4** and run top to bottom. Two FP32 model copies need about 8.8 GB of weights. Save the C4 JSON/PNGs before replacing its runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test first

B4/B8 already had manually packed examples. D1 makes pack/split reusable and validates routing, ownership, positions, device/dtype and boundaries. C3 regression tests must still reject a packed contiguous cache.

Run B4/B8/C3/D1 tests before demos. A failed test stops the notebook. CUDA tests should run on the T4.

In [ ]:
import gc
import torch
from tinyserve.batching import pack_prompts, split_packed
from tinyserve.models.attention import AttentionMetadata, reference_attention
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_packed_prompts
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b4_attention.py",
                           "tests/test_b8_model.py", "tests/test_c3_cache.py", "tests/test_d1_batching.py",
                           "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Flat index is not token position

Take A with three tokens and B with two. Flattening concatenates their IDs, but each request restarts its own positions:

```
flat index:    0  1  2  3  4
tokens:       A0 A1 A2 B0 B1
positions:    0  1  2  0  1
sequence IDs: 0  0  0  1  1
lengths:      (3, 2)
```

Positions feed RoPE and causal ordering. IDs tell attention who owns a token. Lengths split output rows back to requests. These are different jobs.

Five packed tokens versus six padded slots does not prove speedup: our reference still computes dense 5-by-5 attention scores per query head, then masks cross-request pairs.

In [ ]:
hand_prompts = [torch.tensor([1, 7, 9], device=DEVICE), torch.tensor([1, 12], device=DEVICE)]
hand_ids, hand_metadata, hand_lengths = pack_prompts(hand_prompts)
hand_mask = hand_metadata.allowed_mask()
print("IDs:", hand_ids.tolist())
print("positions:", hand_metadata.positions.tolist())
print("sequence IDs:", hand_metadata.sequence_ids.tolist())
print("lengths:", hand_lengths)
print("visible pairs (1 means visible):\n", hand_mask.int().cpu())
assert hand_metadata.positions.tolist() == [0, 1, 2, 0, 1]
assert not bool(hand_mask[:3, 3:].any()) and not bool(hand_mask[3:, :3].any())
hand_layout = {"ids": hand_ids.tolist(), "positions": hand_metadata.positions.tolist(),
               "sequence_ids": hand_metadata.sequence_ids.tolist(), "lengths": list(hand_lengths),
               "allowed_mask": hand_mask.int().tolist()}


## 3. Why resetting positions alone leaks information

Both A0 and B0 have position zero. A position-only mask lets them see each other. Correct visibility requires **same request AND key position <= query position**.

Use three hand tokens: A has two values of 1; B has one value of 10. Set queries to 1 and keys to 0, so every allowed score is equal and softmax takes a simple average.

Correct output is `[1, 1, 10]`. Without sequence IDs, A0 and B0 each average 1 and 10, giving 5.5. This deliberate error uses tiny synthetic attention only; real prompts must remain isolated.

In [ ]:
def demonstrate_missing_request_ids(device):
    prompts = [torch.tensor([1, 2], device=device), torch.tensor([3], device=device)]
    flat_ids, metadata, lengths = pack_prompts(prompts)
    values = torch.tensor([[[1.0]], [[1.0]], [[10.0]]], device=flat_ids.device)
    queries = torch.ones_like(values)
    keys = torch.zeros_like(values)
    correct, weights = reference_attention(queries, keys, values, metadata.allowed_mask())
    wrong, _ = reference_attention(queries, keys, values, AttentionMetadata(metadata.positions).allowed_mask())
    torch.testing.assert_close(correct[:, 0, 0], values[:, 0, 0], rtol=0, atol=0)
    assert abs(wrong[0, 0, 0].item() - 5.5) < 1e-6
    assert torch.count_nonzero(weights[:, ~metadata.allowed_mask()]).item() == 0
    return {"correct_outputs": correct[:, 0, 0].tolist(), "without_request_ids": wrong[:, 0, 0].tolist()}

leakage_demo = demonstrate_missing_request_ids(DEVICE)
print(leakage_demo)


## 4. Real TinyLlama: pack three different prompts

Load ours and HF in FP32. The model still takes `(flat_ids, positions, AttentionMetadata)`; no batch dimension or new model forward is added.

Every token row gets a logits row. `compare_packed_prompts` checks all of them against separate ours/HF forwards, with B8's FP32 tolerance and independent argmax equality assertions.

The model context limit concerns each request, not the sum of unrelated lengths. Dense attention can still run out of memory for large total batches.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
prompt_texts = ["The capital of France is", "The capital of Germany is",
                "A tiny language model can help people by"]
prompt_tensors = [encode(tokenizer, text, DEVICE) for text in prompt_texts]
parity_result = compare_packed_prompts(our_model, hf_model, prompt_tensors)
print("request lengths:", parity_result["request_lengths"])
print("real token rows:", parity_result["total_tokens"], "padded slots:", parity_result["padded_token_slots"])
for row in parity_result["requests"]:
    print(f"request={row['request_index']} tokens={row['tokens']} "
          f"diff(separate)={row['packed_vs_separate_max_abs_diff']:.2e} "
          f"diff(HF)={row['packed_vs_hf_max_abs_diff']:.2e} all argmax equal=True")


## 5. Route one next-token prediction per request

For lengths `(3, 2)`, A uses logits row 2 and B uses row 4. Selecting only the final packed row would answer only B.

Split by lengths, then take each chunk's last row. Those chunks are views into the packed output. This gives one next-token prediction per request, not a complete multi-request generation loop.

Free HF before the remaining demos. Packed no-cache remains compatible with our existing model; the C3 single-request generator is unchanged.

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()
flat_ids, packed_metadata, request_lengths = pack_prompts(prompt_tensors)
with torch.no_grad():
    packed_output = our_model(flat_ids, packed_metadata.positions, packed_metadata)
request_logits = split_packed(packed_output, request_lengths)
predictions = []
for index, (text, logits) in enumerate(zip(prompt_texts, request_logits)):
    next_token_id = logits[-1].argmax().item()
    assert next_token_id == parity_result["requests"][index]["next_token_id"]
    next_text = tokenizer.decode([next_token_id])
    predictions.append({"request_index": index, "prompt": text, "next_token_id": next_token_id,
                        "next_token_text": next_text, "logits_shape": list(logits.shape)})
    print(repr(text), "next token:", repr(next_text), "logits shape:", tuple(logits.shape))


## 6. Verify isolation and ownership after reordering

Change only the second prompt's token content, keeping lengths unchanged. Every output row of the first and third requests must stay unchanged. This checks attention isolation through all 22 layers, not just the mask.

Then reorder requests. Flat indices and batch-local IDs change; the requests themselves do not. Split and route outputs by the new order, then compare with their originals.

A cache with one `length` cannot track these independent requests. C3 packed cached forwards still raise; D2 introduces block tables and slot mappings.

In [ ]:
changed_prompts = list(prompt_tensors)
changed_prompts[1] = (prompt_tensors[1] + 1) % our_model.config.vocab_size
changed_ids, changed_metadata, changed_lengths = pack_prompts(changed_prompts)
with torch.no_grad():
    changed_output = our_model(changed_ids, changed_metadata.positions, changed_metadata)
changed_chunks = split_packed(changed_output, changed_lengths)
isolation_diffs = [(request_logits[index] - changed_chunks[index]).abs().max().item() for index in (0, 2)]
assert isolation_diffs == [0.0, 0.0], "another request changed when request 1 was modified"
order = [2, 0, 1]
reordered_ids, reordered_metadata, reordered_lengths = pack_prompts([prompt_tensors[index] for index in order])
with torch.no_grad():
    reordered_output = our_model(reordered_ids, reordered_metadata.positions, reordered_metadata)
reordered_chunks = split_packed(reordered_output, reordered_lengths)
reordering_diffs = []
for chunk, original_index in zip(reordered_chunks, order):
    torch.testing.assert_close(chunk, request_logits[original_index], rtol=1e-5, atol=1e-4)
    reordering_diffs.append((chunk - request_logits[original_index]).abs().max().item())
print("unchanged request differences:", isolation_diffs)
print("reordering differences:", reordering_diffs)
packed_metadata.cache = our_model.new_kv_cache(flat_ids.numel())
try:
    with torch.no_grad():
        our_model(flat_ids, packed_metadata.positions, packed_metadata)
except ValueError as error:
    cache_error = str(error)
    print("expected packed-cache error:", cache_error)
else:
    raise AssertionError("C3 contiguous cache must reject packed requests")
finally:
    packed_metadata.cache = None


## 7. Save evidence and three interview questions

1. **Why separate positions from request IDs?** Positions control RoPE/causality; IDs isolate ownership.
2. **Why not use only the final packed logits row?** It belongs to only the last request; each request needs its own last row.
3. **Does a block-diagonal mask skip cross-request multiplications?** Not in this dense reference. It prevents visibility, not all computation.

Save the JSON and notebook **with outputs**. Numerical ranges are correctness observations, not timing trials. **Next after D1 validation: D2, block tables, slot mapping and paged cache.**

In [ ]:
separate_diffs = [row["packed_vs_separate_max_abs_diff"] for row in parity_result["requests"]]
hf_diffs = [row["packed_vs_hf_max_abs_diff"] for row in parity_result["requests"]]
saved_path = save_result("D1_flattened_batching", {
    "dtype": "float32", "use_cache": False, "dense_reference_attention": True,
    "hand_layout": hand_layout, "missing_request_ids_demo": leakage_demo,
    "packed_parity": parity_result, "next_token_predictions": predictions,
    "packed_vs_separate_diff_range": {"min": min(separate_diffs), "max": max(separate_diffs)},
    "packed_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "cross_request_isolation_diffs": isolation_diffs,
    "request_reordering": {"order": order, "max_abs_diffs": reordering_diffs},
    "packed_contiguous_cache_error": cache_error,
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del request_logits, packed_output, changed_chunks, changed_output, reordered_chunks, reordered_output
del logits, chunk, our_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
